# M1-INF4 — Python Workshop for Data Analysis
## Session 3 — Cleaning and validating data

A table can load successfully and still produce misleading results. Before analysing it, check what a row means, which values are valid, and which information is missing.

**Learning objectives**
- Audit missing values, data types and duplicate observations.
- Standardise text and convert numbers and dates while keeping track of failed conversions.
- Apply explicit validity rules and justify how missing values are handled.
- Build a reproducible cleaning pipeline, verify its result and export it.

You will reuse selection, Boolean masks, calculated columns, sorting and `groupby()` from the previous notebook.


### Working files
Place this notebook in `notebooks/`, next to the `data/` folder containing `temperatures.csv`. Run cells from top to bottom. Keep the original CSV unchanged.

The small training batch used below is embedded in the notebook; no additional download is required.


In [9]:
from pathlib import Path
import pandas as pd

# Works when the working directory is notebooks/ or the course root.
DATA_DIR = Path("../data") if Path("../data/temperatures.csv").exists() else Path("data")
DATA_FILE = DATA_DIR / "temperatures.csv"
if not DATA_FILE.exists():
    raise FileNotFoundError("Place temperatures.csv in the course data/ folder.")

# The first CSV column is an exported row index, not a measurement.
raw = pd.read_csv(DATA_FILE, index_col=0)
display(raw.head())


,date,city,country,avg_temp_c
0,2000-01-01,Abidjan,Côte D'Ivoire,27.293
1,2000-02-01,Abidjan,Côte D'Ivoire,27.685
2,2000-03-01,Abidjan,Côte D'Ivoire,29.061
3,2000-04-01,Abidjan,Côte D'Ivoire,28.162
4,2000-05-01,Abidjan,Côte D'Ivoire,27.547


## 1. Start with a data-quality audit

The table contains historical **monthly average temperatures**. One observation describes a city in a country for a month.

| Column | Meaning | Expected form |
|---|---|---|
| `date` | Month of observation | A valid date, recorded on the first day of the month |
| `city` | City name | Non-missing text |
| `country` | Country name | Non-missing text |
| `avg_temp_c` | Monthly average temperature | A number in degrees Celsius, or a missing value |

The proposed observation key is `country`, `city`, `date`. An index is a row label; it is not automatically a meaningful identifier.

A negative Celsius temperature is valid. A missing temperature is not zero.


In [10]:
print("Rows and columns:", raw.shape)
raw.info()
display(raw.describe(include="all"))


Rows and columns: (16500, 4)
<class 'pandas.DataFrame'>
RangeIndex: 16500 entries, 0 to 16499
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        16500 non-null  str    
 1   city        16500 non-null  str    
 2   country     16500 non-null  str    
 3   avg_temp_c  16407 non-null  float64
dtypes: float64(1), str(3)
memory usage: 515.8 KB


,date,city,country,avg_temp_c
count,16500,16500,16500,16407.000000
unique,165,100,49,NaN
top,2000-01-01,Abidjan,China,NaN
freq,100,165,2640,NaN
mean,NaN,NaN,NaN,19.732877
std,NaN,NaN,NaN,9.579635
min,NaN,NaN,NaN,-23.495000
25%,NaN,NaN,NaN,14.870000
50%,NaN,NaN,NaN,22.119000
75%,NaN,NaN,NaN,27.001500


### Counting missing values
`isna()` produces a Boolean table. For each column, `sum()` counts `True` values and `mean()` gives their proportion, because `True` behaves like 1 and `False` like 0 in these aggregations.


In [11]:
quality = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_pct": raw.isna().mean() * 100,
    "dtype": raw.dtypes.astype(str)
})
display(quality.sort_values("missing_pct", ascending=False))


,missing_count,missing_pct,dtype
avg_temp_c,93,0.563636,float64
date,0,0.000000,str
city,0,0.000000,str
country,0,0.000000,str


### Your turn — What needs attention?
1. Display only rows with a missing temperature.
2. Count missing temperatures by city, in descending order. Hint: select the missing rows first, then use `groupby("city").size()`.
3. Calculate the percentage of missing temperatures in the whole table.
4. Explain why loading the file without an error does not mean the data is ready for analysis.


In [12]:
# Your code here
missing_temp = raw[raw["avg_temp_c"].isna()]
display(missing_temp)

missing_by_city = missing_temp.groupby("city").size().sort_values(ascending=False)
display(missing_by_city)

pct_missing = raw["avg_temp_c"].isna().mean() * 100
print(f"Missing avg_temp_c: {pct_missing:.2f}%")

,date,city,country,avg_temp_c
164,2013-09-01,Abidjan,Côte D'Ivoire,NaN
329,2013-09-01,Addis Abeba,Ethiopia,NaN
494,2013-09-01,Ahmadabad,India,NaN
659,2013-09-01,Aleppo,Syria,NaN
824,2013-09-01,Alexandria,Egypt,NaN
...,...,...,...,...
15674,2013-09-01,Tianjin,China,NaN
15839,2013-09-01,Tokyo,Japan,NaN
16169,2013-09-01,Umm Durman,Sudan,NaN
16334,2013-09-01,Wuhan,China,NaN


city
Abidjan        1
Addis Abeba    1
Ahmadabad      1
Aleppo         1
Alexandria     1
              ..
Tianjin        1
Tokyo          1
Umm Durman     1
Wuhan          1
Xian           1
Length: 93, dtype: int64

Missing avg_temp_c: 0.56%


Loading without an error only means pandas could parse the file into rows and columns. It says nothing about whether values are missing, mistyped, duplicated or out of a valid range — those need a separate check.

## 2. Standardise representations before making decisions

Work on a copy. Keep the raw values available so you can explain what changed.

### 2.1 Text labels
Whitespace and inconsistent case can split one category into several groups. `.str.strip()` removes leading and trailing whitespace; `.str.title()` standardises case for this exercise. In other datasets, acronyms and official spellings may require an explicit mapping instead.


In [13]:
labels = pd.Series([" London ", "LONDON", "Paris", None], dtype="string")
display(pd.DataFrame({"before": labels, "after": labels.str.strip().str.title()}))


,before,after
0,London,London
1,LONDON,London
2,Paris,Paris
3,<NA>,<NA>


### 2.2 Numbers stored as text
`pd.to_numeric(..., errors="coerce")` converts values it can read and replaces invalid values with a missing value. It does not recover the correct measurement.

Here the input convention explicitly uses a comma as a decimal separator. Replace it with a dot before conversion; do not generalise this rule to thousands separators without checking the file specification.


In [14]:
text_values = pd.Series(["12.5", "8,2", "unknown", None], dtype="string")
normalised_values = text_values.str.replace(",", ".", regex=False)
numeric_values = pd.to_numeric(normalised_values, errors="coerce")
failed_conversion = text_values.notna() & numeric_values.isna()
display(pd.DataFrame({
    "before": text_values,
    "after": numeric_values,
    "conversion_failed": failed_conversion
}))


,before,after,conversion_failed
0,12.5,12.5,False
1,"8,2",8.2,False
2,unknown,<NA>,True
3,<NA>,<NA>,False


### 2.3 Dates are more than text
`pd.to_datetime()` checks calendar validity. With `errors="coerce"`, an invalid date becomes `NaT`, the datetime missing value. Use the documented input format rather than guessing whether a date is day-first or month-first.


In [15]:
date_text = pd.Series(["2001-01-01", "2001-02-30", None])
parsed_dates = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
display(pd.DataFrame({"before": date_text, "after": parsed_dates}))
display(parsed_dates.dt.year)


,before,after
0,2001-01-01,2001-01-01
1,2001-02-30,NaT
2,NaN,NaT


0    2001.0
1       NaN
2       NaN
dtype: float64

### Your turn — Convert, then explain
A sensor export contains the strings below. Convert the values to numbers and identify **newly created** missing values. Explain why counting all missing values after conversion would not give the same answer.

```python
sensor = pd.Series(["-2,5", "0", "missing", None], dtype="string")
```


In [16]:
# Your code here
sensor = pd.Series(["-2,5", "0", "missing", None], dtype="string")

sensor_numeric = pd.to_numeric(sensor.str.replace(",", ".", regex=False), errors="coerce")
newly_missing = sensor.notna() & sensor_numeric.isna()

display(pd.DataFrame({"before": sensor, "after": sensor_numeric, "newly_missing": newly_missing}))
print("Newly created missing values:", newly_missing.sum())
print("Total missing after conversion:", sensor_numeric.isna().sum())

,before,after,newly_missing
0,"-2,5",-2.5,False
1,0,0.0,False
2,missing,<NA>,True
3,<NA>,<NA>,False


Newly created missing values: 1
Total missing after conversion: 2


Counting all missing values after conversion mixes two different things: the value that was already missing (`None`), and the value that only became missing because it failed to convert (`"missing"`). `newly_missing` isolates the second group, which is what actually tells you the conversion introduced a problem.

## 3. Group problem 1 — Is this incoming batch ready?

The following batch is derived from a few London and Paris records. **Some values have deliberately been altered for this training exercise, and one row has been copied.** These issues must not be interpreted as defects in the original dataset.

Run the cell to create `incoming`. This is a separate object from `raw`.


In [17]:
incoming = pd.DataFrame(
[{'date': '2000-01-01',
  'city': ' london ',
  'country': 'United Kingdom',
  'avg_temp_c': 4.6930000000000005},
 {'date': '2000-02-01',
  'city': 'London',
  'country': 'United Kingdom',
  'avg_temp_c': '6,115'},
 {'date': '2000-03-01',
  'city': 'London',
  'country': 'United Kingdom',
  'avg_temp_c': 'not recorded'},
 {'date': '2000-02-30',
  'city': 'London',
  'country': 'United Kingdom',
  'avg_temp_c': 8.246},
 {'date': '2000-05-01',
  'city': 'London',
  'country': 'United Kingdom',
  'avg_temp_c': '999'},
 {'date': '2000-06-01',
  'city': 'London',
  'country': ' united kingdom ',
  'avg_temp_c': 15.875},
 {'date': '2000-01-01', 'city': ' PARIS ', 'country': 'France', 'avg_temp_c': 3.845},
 {'date': '2000-02-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': None},
 {'date': '2000-03-01',
  'city': 'Paris',
  'country': 'FRANCE ',
  'avg_temp_c': 7.872000000000001},
 {'date': '2000-04-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': 10.067},
 {'date': '2000-05-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': 15.451},
 {'date': '2000-06-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': 17.666},
 {'date': '2000-04-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': 10.067}]
)
display(incoming)


,date,city,country,avg_temp_c
0,2000-01-01,london,United Kingdom,4.693
1,2000-02-01,London,United Kingdom,"6,115"
2,2000-03-01,London,United Kingdom,not recorded
3,2000-02-30,London,United Kingdom,8.246
4,2000-05-01,London,United Kingdom,999
5,2000-06-01,London,united kingdom,15.875
6,2000-01-01,PARIS,France,3.845
7,2000-02-01,Paris,France,None
8,2000-03-01,Paris,FRANCE,7.872
9,2000-04-01,Paris,France,10.067


### Batch specification
- City and country labels must be stripped of surrounding whitespace and use title case.
- Dates use `YYYY-MM-DD` and must be valid first days of a month.
- Temperatures use Celsius. A comma may be a decimal separator.
- For **this exercise**, temperatures outside **−50 to 60 °C** require review; this is a screening rule, not a universal scientific validity range.
- There should be at most one observation per country, city and month.

### Your investigation
1. Preserve `incoming` and create `prepared = incoming.copy()`.
2. Audit missing values, types and exact duplicate rows **before** changing anything.
3. Standardise city/country text and convert temperatures and dates.
4. Display the original rows whose non-missing temperatures or dates could not be converted.
5. Display numeric temperatures outside the screening range.
6. Propose a policy for each issue: correct a representation, retain a missing value, remove a confirmed duplicate, or set a row aside for review. **Do not delete rows yet.**

**Group deliverable:** code, a small issue report, and a short explanation of your proposed policy. Each member must be able to explain the masks used.


### Your group answer


In [18]:
# Your code here
prepared = incoming.copy()

# 2. audit before changing anything
print("Shape:", prepared.shape)
prepared.info()
print("Missing values:")
print(prepared.isna().sum())
print("Exact duplicate rows:", prepared.duplicated().sum())

# 3. standardise text, convert temperatures and dates (keep originals for the audit)
prepared["city"] = prepared["city"].str.strip().str.title()
prepared["country"] = prepared["country"].str.strip().str.title()

prepared["avg_temp_c_raw"] = prepared["avg_temp_c"]
temp_as_text = prepared["avg_temp_c"].astype("string").str.replace(",", ".", regex=False)
prepared["avg_temp_c"] = pd.to_numeric(temp_as_text, errors="coerce")

prepared["date_raw"] = prepared["date"]
prepared["date"] = pd.to_datetime(prepared["date"], format="%Y-%m-%d", errors="coerce")

# 4. rows whose non-missing original value could not be converted
temp_conversion_failed = prepared["avg_temp_c_raw"].notna() & prepared["avg_temp_c"].isna()
date_conversion_failed = prepared["date_raw"].notna() & prepared["date"].isna()
display(prepared.loc[temp_conversion_failed | date_conversion_failed])

# 5. numeric temperatures outside the screening range
out_of_range = prepared["avg_temp_c"].notna() & ~prepared["avg_temp_c"].between(-50, 60)
display(prepared.loc[out_of_range])

Shape: (13, 4)
<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        13 non-null     str   
 1   city        13 non-null     str   
 2   country     13 non-null     str   
 3   avg_temp_c  12 non-null     object
dtypes: object(1), str(3)
memory usage: 548.0+ bytes
Missing values:
date          0
city          0
country       0
avg_temp_c    1
dtype: int64
Exact duplicate rows: 1


,date,city,country,avg_temp_c,avg_temp_c_raw,date_raw
2,2000-03-01,London,United Kingdom,<NA>,not recorded,2000-03-01
3,NaT,London,United Kingdom,8.246,8.246,2000-02-30


,date,city,country,avg_temp_c,avg_temp_c_raw,date_raw
4,2000-05-01,London,United Kingdom,999.0,999,2000-05-01


### Your group answer

- Whitespace/case differences (" london ", "LONDON", " united kingdom ", "FRANCE ") are representation issues, fixed by standardising text.
- `"not recorded"` fails numeric conversion and becomes a missing temperature — keep the row, keep the value missing.
- `"999"` converts fine but falls outside -50/60°C — set aside for review as a likely sensor error, not deleted.
- `2000-02-30` is not a valid calendar date, so it fails conversion — set aside for review, not guessed at.
- The last Paris row (2000-04-01) is an exact copy of an earlier row — a confirmed duplicate, safe to drop one copy.

## 4. Duplicates: identical rows or conflicting observations?

`duplicated()` compares whole rows by default. `duplicated(subset=key, keep=False)` identifies **all** rows sharing an observation key. A repeated key may contain conflicting measurements, so it is not automatically safe to keep the first row.


In [19]:
example = pd.DataFrame({
    "city": ["London", "London", "Paris", "Paris"],
    "date": ["2000-01-01"] * 4,
    "avg_temp_c": [4.7, 4.7, 5.0, 7.0]
})
print("Extra identical rows:", example.duplicated().sum())
unique_rows = example.drop_duplicates().copy()
conflicts = unique_rows.duplicated(subset=["city", "date"], keep=False)
display(unique_rows.loc[conflicts])


Extra identical rows: 1


,city,date,avg_temp_c
2,Paris,2000-01-01,5.0
3,Paris,2000-01-01,7.0


The London rows are identical; if confirmed as a repeated import, keep one. The Paris values disagree for the same key. Set them aside for review instead of silently choosing a temperature. In the supplied monthly data, use `country`, `city`, `date` together as the key.


## 5. Missing values: choose a policy that fits the question

- Missing observation keys prevent identification: set these rows aside for review.
- Missing measurements can be retained and counted. pandas `mean()` skips missing measurements by default.
- `dropna(subset=[...])` removes rows missing the listed fields. Plain `dropna()` may remove more rows than intended.
- `fillna(0)` asserts a value of zero. For temperature, zero is a real measurement, not a neutral placeholder.

A result should report both the statistic and how much data supports it.


In [20]:
measurements = pd.Series([10.0, 20.0, float("nan")])
print("Mean of available measurements:", measurements.mean())
print("Mean after inventing a zero:", measurements.fillna(0).mean())
print("Available / total:", measurements.count(), "/", measurements.size)


Mean of available measurements: 15.0
Mean after inventing a zero: 10.0
Available / total: 2 / 3


### An explicit policy for our monthly table
Keep missing temperatures in the cleaned table, and report their absence in summaries. Set invalid-key rows and out-of-range temperatures aside for review. Keep one copy of a confirmed exact duplicate. Set conflicting keys aside rather than choosing arbitrarily.

An empty label should also count as missing after trimming. A temperature that fails numeric conversion becomes missing, but the original value and failure count remain in the audit.


In [21]:
# Small illustration: these labels all mean an absent identifier.
empty_labels = pd.Series(["Paris", "   ", None], dtype="string")
display(empty_labels.str.strip().replace("", pd.NA))


0    Paris
1     <NA>
2     <NA>
dtype: string

### Toolkit — Combining checks and validating results

- `table[key].isna().any(axis=1)` flags a row if **any** key field is missing. `axis=1` means “across columns, for each row”.
- `table["date"].dt.day` extracts the day of a parsed date; `.ne(1)` means “not equal to 1”.
- `values.between(-50, 60)` includes both endpoints. Check `notna()` separately to distinguish missing values from out-of-range numbers.
- `table.loc[mask].copy()` creates a separate table for flagged rows. Assign a string column such as `review["reason"] = "invalid date"` to document a decision.
- In a grouped summary, `size` counts rows and `count` counts non-missing values in the selected column.
- `assert condition` stops execution if a promised property is false. For example, `assert len(before) == len(kept) + len(rejected)` checks row accounting when no duplicate copies have been removed. Include those copies in the accounting when necessary.

Checks support a documented policy; they do not prove that every retained measurement is correct.


## 6. Group problem 2 — Deliver an auditable clean table

Build a reproducible pipeline starting again from `incoming.copy()`. Do not rely on someone having completed problem 1 correctly.

Apply the policy above:
1. Standardise labels; treat empty labels as missing. Convert numbers and dates.
2. Remove confirmed exact duplicate rows after standardisation. Record how many copies were removed.
3. Set aside rows with a missing country/city/date, a date that is not the first of its month, or a non-missing temperature outside −50 to 60 °C.
4. Among the remaining rows, set aside **all** rows with conflicting observation keys.
5. Keep the other rows, including missing temperatures, as `clean_batch`. Keep rejected records in `review_batch` and add a reason column.
6. Produce a per-city summary with total rows, available temperatures and mean temperature. Explain the difference between `size` and `count`.
7. Verify that every input row is accounted for: input rows = clean rows + review rows + removed duplicate copies.
8. Save the clean table and review table as CSV files with `index=False`.

**Group deliverable:** reproducible code, both tables, the row-accounting check, and three sentences explaining your decisions and their impact on the mean. Rotate who writes and who reviews the code.

The export filenames are `temperatures_batch_clean.csv` and `temperatures_batch_review.csv`. Keep them in a `cleaned/` folder next to this notebook.


### Your group answer


In [22]:
# Your code here
start = incoming.copy()
n_input = len(start)

# 1. standardise labels (empty label counts as missing), convert numbers and dates
start["city"] = start["city"].str.strip().str.title().replace("", pd.NA)
start["country"] = start["country"].str.strip().str.title().replace("", pd.NA)

temp_as_text = start["avg_temp_c"].astype("string").str.replace(",", ".", regex=False)
start["avg_temp_c"] = pd.to_numeric(temp_as_text, errors="coerce")

start["date"] = pd.to_datetime(start["date"], format="%Y-%m-%d", errors="coerce")

# 2. remove confirmed exact duplicate rows after standardisation
before_dedup = len(start)
start = start.drop_duplicates()
removed_duplicates = before_dedup - len(start)
print("Exact duplicate copies removed:", removed_duplicates)

# 3. set aside rows with a missing key, a date not on the 1st, or an out-of-range temperature
missing_key = start[["country", "city", "date"]].isna().any(axis=1)
not_first_of_month = start["date"].notna() & start["date"].dt.day.ne(1)
out_of_range = start["avg_temp_c"].notna() & ~start["avg_temp_c"].between(-50, 60)

reason = pd.Series(pd.NA, index=start.index, dtype="string")
reason[missing_key] = "missing key"
reason[not_first_of_month & reason.isna()] = "invalid date"
reason[out_of_range & reason.isna()] = "out of range temperature"

set_aside_mask = missing_key | not_first_of_month | out_of_range
candidates = start.loc[~set_aside_mask].copy()
review_stage1 = start.loc[set_aside_mask].copy()
review_stage1["reason"] = reason[set_aside_mask]

# 4. among the remaining rows, set aside all rows with conflicting observation keys
conflict_mask = candidates.duplicated(subset=["country", "city", "date"], keep=False)
review_stage2 = candidates.loc[conflict_mask].copy()
review_stage2["reason"] = "conflicting key"
clean_batch = candidates.loc[~conflict_mask].copy()

# 5. combine rejected records
review_batch = pd.concat([review_stage1, review_stage2], ignore_index=True)

# 6. per-city summary: size counts all rows, count only non-missing temperatures
summary = clean_batch.groupby("city")["avg_temp_c"].agg(["size", "count", "mean"])
summary = summary.rename(columns={"size": "total_rows", "count": "available_temps", "mean": "mean_temp"})
display(summary)

# 7. row accounting
assert n_input == len(clean_batch) + len(review_batch) + removed_duplicates
print("Row accounting OK:", n_input, "=", len(clean_batch), "+", len(review_batch), "+", removed_duplicates)

# 8. export
OUTPUT_DIR = Path("cleaned")
OUTPUT_DIR.mkdir(exist_ok=True)
clean_batch.to_csv(OUTPUT_DIR / "temperatures_batch_clean.csv", index=False)
review_batch.to_csv(OUTPUT_DIR / "temperatures_batch_review.csv", index=False)

Exact duplicate copies removed: 1


,total_rows,available_temps,mean_temp
city,,,
London,4,3,8.894333
Paris,6,5,10.9802


Row accounting OK: 13 = 10 + 2 + 1


### Your group answer

Setting temperatures aside instead of deleting them keeps the row count auditable: every input row ends up in `clean_batch`, `review_batch`, or the duplicate count, and the assert confirms nothing was silently lost. Keeping missing temperatures in `clean_batch` (instead of dropping or zero-filling them) means the per-city mean is computed only from real measurements, since `count` already tells us how many that is. Rows flagged for a conflicting key are the ones that would otherwise have silently biased the mean, because we would have had to guess which of two disagreeing values was correct.

## 7. Transfer to the complete dataset

Return to the original `raw` table. The training issues were deliberately introduced; do not assume they occur in the full CSV.

Your aim is to prepare the table for exploratory analysis while documenting its remaining limitations.


### Your turn — Ready for exploration?
1. Starting from `raw.copy()`, convert dates and check the monthly observation key, exact duplicates and the screening range.
2. Retain missing temperatures. Confirm how many are missing, without replacing them with zero.
3. For each country and city, report total observations, available temperatures, mean temperature and missing percentage.
4. Export `temperatures_clean.csv` with `index=False` into `cleaned/`.
5. Explain why cities with different time coverage should not be compared using their overall means alone.

You may reuse your pipeline from problem 2. If you did not finish it, write these steps directly for the full table.


In [23]:
# Your code here
clean_full = raw.copy()
clean_full["date"] = pd.to_datetime(clean_full["date"], format="%Y-%m-%d", errors="coerce")

# 1. key / exact duplicates / screening range checks
missing_key = clean_full[["country", "city", "date"]].isna().any(axis=1)
not_first_of_month = clean_full["date"].notna() & clean_full["date"].dt.day.ne(1)
exact_duplicates = clean_full.duplicated().sum()
out_of_range = clean_full["avg_temp_c"].notna() & ~clean_full["avg_temp_c"].between(-50, 60)

print("Rows with a missing key:", missing_key.sum())
print("Rows not on the 1st of the month:", not_first_of_month.sum())
print("Exact duplicate rows:", exact_duplicates)
print("Rows outside the -50/60 screening range:", out_of_range.sum())

# 2. missing temperatures, retained (not replaced with zero)
missing_temp_count = clean_full["avg_temp_c"].isna().sum()
print("Missing temperatures kept as NaN:", missing_temp_count)

# 3. per country/city summary
summary_full = clean_full.groupby(["country", "city"])["avg_temp_c"].agg(["size", "count", "mean"])
summary_full = summary_full.rename(columns={"size": "total_rows", "count": "available_temps", "mean": "mean_temp"})
summary_full["missing_pct"] = (1 - summary_full["available_temps"] / summary_full["total_rows"]) * 100
display(summary_full)

# 4. export
OUTPUT_DIR = Path("cleaned")
OUTPUT_DIR.mkdir(exist_ok=True)
clean_full.to_csv(OUTPUT_DIR / "temperatures_clean.csv", index=False)

Rows with a missing key: 0
Rows not on the 1st of the month: 0
Exact duplicate rows: 0
Rows outside the -50/60 screening range: 0
Missing temperatures kept as NaN: 93


total_rows  available_temps  mean_temp  \
country       city                                                       
Afghanistan   Kabul                    165              164  15.525756   
Angola        Luanda                   165              164  24.387659   
Australia     Melbourne                165              164  14.264043   
              Sydney                   165              164  17.792165   
Bangladesh    Dhaka                    165              164  26.164378   
...                                    ...              ...        ...   
United States Chicago                  165              165  11.326170   
              Los Angeles              165              165  16.649121   
              New York                 165              165  10.888255   
Vietnam       Ho Chi Minh City         165              164  27.909878   
Zimbabwe      Harare                   165              164  20.721988   

                                missing_pct  
country       city                           
Afghanistan   Kabul                0.606061  
Angola        Luanda               0.606061  
Australia     Melbourne            0.606061  
              Sydney               0.606061  
Bangladesh    Dhaka                0.606061  
...                                     ...  
United States Chicago              0.000000  
              Los Angeles          0.000000  
              New York             0.000000  
Vietnam       Ho Chi Minh City     0.606061  
Zimbabwe      Harare               0.606061  

[100 rows x 4 columns]

A city with only a few recorded months and a city with decades of data can land on the same overall mean by coincidence, but that mean rests on very different amounts of evidence and can hide seasonal swings the shorter series never captured. Comparing them fairly means checking `total_rows`/`available_temps` alongside the mean, not the mean alone.

### Check the exported file
CSV stores values and column names, but does not preserve all pandas types. Parse dates again when loading it. `index=False` prevents an unintended extra index column.


In [24]:
# Run after completing the export above.
OUTPUT_DIR = Path("cleaned")
export_file = OUTPUT_DIR / "temperatures_clean.csv"
if export_file.exists():
    reloaded = pd.read_csv(export_file, parse_dates=["date"])
    display(reloaded.head())
    reloaded.info()
else:
    print("Complete the export exercise to create temperatures_clean.csv.")


,date,city,country,avg_temp_c
0,2000-01-01,Abidjan,Côte D'Ivoire,27.293
1,2000-02-01,Abidjan,Côte D'Ivoire,27.685
2,2000-03-01,Abidjan,Côte D'Ivoire,29.061
3,2000-04-01,Abidjan,Côte D'Ivoire,28.162
4,2000-05-01,Abidjan,Côte D'Ivoire,27.547


<class 'pandas.DataFrame'>
RangeIndex: 16500 entries, 0 to 16499
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        16500 non-null  datetime64[us]
 1   city        16500 non-null  str           
 2   country     16500 non-null  str           
 3   avg_temp_c  16407 non-null  float64       
dtypes: datetime64[us](1), float64(1), str(2)
memory usage: 515.8 KB


## 8. OPTIONAL / IF TIME ALLOWS — Categories and memory

Repeated labels can be stored with the `category` dtype. This changes storage and type semantics; it does not clean invalid labels or make a number meaningful.

A numeric identifier such as a store code is not automatically a quantity. Similarly, do not convert arbitrary text such as `"False"` to Boolean with `astype(bool)`: non-empty strings are truthy. Use an explicit mapping when values encode yes/no.


In [25]:
memory_demo = raw.copy()
before = memory_demo.memory_usage(deep=True).sum()
for col in ["city", "country"]:
    memory_demo[col] = memory_demo[col].astype("category")
after = memory_demo.memory_usage(deep=True).sum()
print("Before (bytes):", before)
print("After (bytes):", after)
print("Reduction (%):", round((before - after) / before * 100, 1))


Before (bytes): 2977887
After (bytes): 1147165
Reduction (%): 61.5


## Key takeaways

- Diagnose problems before dropping rows or filling values.
- Distinguish a representation fix from an invented measurement.
- Preserve raw data, record exclusions and count every row.
- Check both identical rows and conflicts on the observation key.
- Validate the cleaned table against explicit rules.
- Report missingness and coverage alongside statistics.

**Next:** explore distributions, compare groups and visualise data, using the quality checks to interpret what the figures can and cannot show.